## **Music Recommendation Algorithm Project**
</br>
Info here

***Data Transformation***

Details

---
### **1. Imports**

In [ ]:
# Importing sys to ensure proper environment setup
import sys

print(sys.version_info)

sys.version_info(major=3, minor=11, micro=14, releaselevel='final', serial=0)


In [ ]:
# Importing pandas and numpy for numerical analysis
# Importing pyplot and seaborn to visualize the data
# Importing os, pathlib, and warnings for functionality, faster loading, flagging exceptions, etc.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import ticker, pylab
from matplotlib.legend import Legend
import statistics
from scipy.stats import skew, kurtosis, trim_mean
import seaborn as sns
import lightgbm as lgb
import os # possibly remove
from pathlib import Path, PureWindowsPath
from scipy.stats import multivariate_normal, norm, trim_mean, zscore
import warnings
warnings.filterwarnings('ignore')

# Importing sklearn items for preprocessing, model training & testing
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler, RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, StratifiedGroupKFold
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.metrics import (
    mean_squared_error, r2_score, mean_absolute_error, classification_report,
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier

# Different label assignment (assign_labels="cluster_qr") as deterministic partitioning alternative
from sklearn.cluster import KMeans, AffinityPropagation, DBSCAN, SpectralClustering
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.decomposition import PCA

# Makes graphs appear in line
%matplotlib inline

sns.set(style="whitegrid", palette="Set3", font_scale=1.25)    # alt muted, spectral, tab20c

print("Setup Complete")

Setup Complete


---
### **2. Load Data & Quick Review**

In [ ]:
# Explicitly noting path as being in Windows format to avoid issues with backsplash
#filename = PureWindowsPath("..Data\tcc_ceds_music.csv")

# Convert path to the correct format
#file_path = Path(filename)
file = open("C:\\Users\\winni\\music-rec-algo\\Data\\tcc_ceds_music.csv")

# Loading data as a DataFrame
df = pd.read_csv(file) 

# Using head() function to display the first five rows of the data
print("Heads")
print(df.head())

print()

# Using tail() function to display the last five rows of the data
print("Tails")
print(df.tail())

Heads
   Unnamed: 0           artist_name            track_name  release_date genre  \
0           0                mukesh  mohabbat bhi jhoothi          1950   pop   
1           4         frankie laine             i believe          1950   pop   
2           6           johnnie ray                   cry          1950   pop   
3          10           pérez prado              patricia          1950   pop   
4          12  giorgos papadopoulos    apopse eida oneiro          1950   pop   

                                              lyrics  len    dating  violence  \
0  hold time feel break feel untrue convince spea...   95  0.000598  0.063746   
1  believe drop rain fall grow believe darkest ni...   51  0.035537  0.096777   
2  sweetheart send letter goodbye secret feel bet...   24  0.002770  0.002770   
3  kiss lips want stroll charm mambo chacha merin...   54  0.048249  0.001548   
4  till darling till matter know till dream live ...   48  0.001350  0.001350   

   world/life  ...  

In [ ]:
# Using shape() function to return a tuple listing number of rows and columns in the data

print("Shape:", df.shape)

Shape: (28372, 31)


In [ ]:
# Using info() function to view column names, data types, and other relevant information

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28372 entries, 0 to 28371
Data columns (total 31 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Unnamed: 0                28372 non-null  int64  
 1   artist_name               28372 non-null  object 
 2   track_name                28372 non-null  object 
 3   release_date              28372 non-null  int64  
 4   genre                     28372 non-null  object 
 5   lyrics                    28372 non-null  object 
 6   len                       28372 non-null  int64  
 7   dating                    28372 non-null  float64
 8   violence                  28372 non-null  float64
 9   world/life                28372 non-null  float64
 10  night/time                28372 non-null  float64
 11  shake the audience        28372 non-null  float64
 12  family/gospel             28372 non-null  float64
 13  romantic                  28372 non-null  float64
 14  commun

In [ ]:
# Updating column names to remove whitespace and erroroneous characters

df.columns = df.columns.str.strip().str.replace(' ', '_').str.replace('(', '').str.replace(':', '')
df.columns = df.columns.str.replace(')', '').str.replace('-', '_').str.replace('/', '_')
print("Updated Column Names:", df.columns)
print(df.info())

Updated Column Names: Index(['Unnamed_0', 'artist_name', 'track_name', 'release_date', 'genre',
       'lyrics', 'len', 'dating', 'violence', 'world_life', 'night_time',
       'shake_the_audience', 'family_gospel', 'romantic', 'communication',
       'obscene', 'music', 'movement_places', 'light_visual_perceptions',
       'family_spiritual', 'like_girls', 'sadness', 'feelings', 'danceability',
       'loudness', 'acousticness', 'instrumentalness', 'valence', 'energy',
       'topic', 'age'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28372 entries, 0 to 28371
Data columns (total 31 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Unnamed_0                 28372 non-null  int64  
 1   artist_name               28372 non-null  object 
 2   track_name                28372 non-null  object 
 3   release_date              28372 non-null  int64  
 4   genre                     28372 non

In [ ]:
# Finding missing values and if relevant, consider altering or dropping

missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)

pd.DataFrame({'count': missing, 'percent': missing_pct})

--- 
### **3. Alterations & Modifications**

|  **Genres**    |  **Features**    |    
| :--- | :---: |   
|  Blues    |  Acousticness    |    
|  Country    |  Danceability    |    
|  Hip Hop    |  Energy    |    
|  Jazz    |  Instrumentalness    |    
|  Pop    |  Loudness    |    
|  Reggae    |  Valence    |    
|  Rock    |        |    

Based on the EDA, altering and modifying dataset to only include relevant information. **Dropping unnecessary columns such as:**    
- ***len*** reason here    
- ***lyrics***  reason here    
- ***topic***  reason here    

In [ ]:
# Dropping irrelevant columns

df1 = df.drop(columns=["len", "lyrics", "topic"])
df1.columns.tolist()

# Updated shape after dropping columns
print("Shape after dropping columns:", df1.shape)

##### **<p style="text-align:center;">3A. Encoding the Genre Column</p>**    
The 'genre' column is text (ie pop) but modeling requires numbers. As such, using encoding methods to convert it. This conversion will create a new column for each genre with a boolean value (0 or 1).

In [ ]:
# Label encoding the genre column to convert categorical data into numerical data for analysis

le = LabelEncoder()
df1['genre_enc'] = le.fit_transform(df1['genre'])

print("Label Encoder - Category Mapping:")
print(dict(enumerate(le.classes_)))

# One-Hot Encoding via get_dummies to drop 'genre' column)
df1 = pd.get_dummies(df1, columns=['genre'], drop_first=True, dtype=int)

print("DF after get_dummies:/n", df1.head(5))

In [ ]:
# Dropping genre_enc column as it is no longer needed after encoding

df1 = df1.drop(columns=['genre_enc'])
print("DF after dropping genre_enc:/n", df1.head(5))

---
### **4. Skews, Outliers & Additional Transformations**

In [ ]:
# Visualizing distributions of skewed columns
skewness = df1.skew(numeric_only=True).sort_values(ascending=False)
print("Skewness:\n", skewness)

skewed_cols = skewness[abs(skewness) > 0.5].index.tolist()

# Exclude target label from transformation --> need to determine
# skewed_cols = [col for col in skewed_cols if col != 'NAME']

fig, axes = plt.subplots(len(skewed_cols), 2, figsize=(12, 4 * len(skewed_cols)))
for i, col in enumerate(skewed_cols):
    df[col].hist(ax=axes[i, 0], bins=30)
    axes[i, 0].set_title(f'{col} - Original')
    np.log1p(df[col].clip(lower=0)).hist(ax=axes[i, 1], bins=30)
    axes[i, 1].set_title(f'{col} - Log Transformed')
plt.tight_layout()
plt.show()

In [ ]:
# Handling outliers with IQR capping at 1.5*IQR boundary (Winsorization)
# Done in lieu of dropping rows, thereby preserving data

for col in skewed_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df1[col] = df1[col].clip(lower=lower, upper=upper)

# Using log1p again to handle zero values safely
# Only appling it to positively skewed columns (skew > 0.5)
for col in skewed_cols:
    if skewness[col] > 0.5:
        df1[col] = np.log1p(df1[col])
    elif skewness[col] < -0.5:
        # Reflect then log for negatively skewed
        df1[col] = np.log1p(df1[col].max() - df1[col])

In [ ]:
# Checking for any missing values
print(df1.isnull().sum())

# For numeric columns, fill with median (robust to outliers)
num_cols = df1.select_dtypes(include='number').columns
df1[num_cols] = df1[num_cols].fillna(df1[num_cols].median())

# For categorical columns, fill with mode
cat_cols = df1.select_dtypes(include='object').columns
for col in cat_cols:
    df1[col] = df1[col].fillna(df1[col].mode()[0])

##### **<p style="text-align:center;">4A. Feature Engineering</p>**  
Observations from the EDA are the foundation of the new features created to help the model recommend. ADD MORE INFO HERE

In [ ]:
# Creating new features based on EDA
# balanceDiffOrg = how much the origin account balance changed
# if this equals the amount and the new balance is 0 that's suspicious
df1['balanceDiffOrg'] = df1['oldbalanceOrg'] - df1['amount']

# balanceDiffDest = how much the destination account balance changed
df1['balanceDiffDest'] = df1['oldbalanceDest'] + df1['amount']

print("Shape after feature engineering:", df.shape)
print("\nNew columns:", df.columns.tolist())
df1.head()

---
### **5. Verifying Changes**

In [ ]:
# Confirm no missing values remain
print("Missing values after cleaning:")
print(df1.isnull().sum())
print()

# Confirm shape is intact
print("DataFrame shape:\n", df1.shape)

# Numeric columns check
print("\nNumeric column stats:")
print(df1[num_cols].describe())

---
### **6. Save Cleaned CSV**

In [ ]:
# Explicity noting path as being in Windows format so I can use forward slash
clean = PureWindowsPath("C:\\Users\\Winni\\music-rec-algo\\Data\\cleaned.csv")

# Convert path to the correct format
file_path = Path(clean)

# Saving data as a DataFrame
df1.to_csv(file_path, index=False)
# index=False prevents pandas from saving row numbers as a column
print("Cleaned data saved successfully!")

In [ ]:
# Most reliable confirmation is to reload saved file and review it
clean = PureWindowsPath("C:\\Users\\Winni\\music-rec-algo\\Data\\cleaned.csv")
file_path = Path(clean)

df1_clean = pd.read_csv(file_path)

print("Shape:", df1_clean.shape)
print("Missing values:\n", df1_clean.isnull().sum())
print("\nFinal columns:", df1_clean.columns.tolist())
print("\nPreview:\n", df1_clean.head(5))